# Broadcast CV pipeline run

Runs roboflow/sports soccer modes on a 20-second broadcast clip, times each mode, and saves outputs for review.

Kernel: Colab with **GPU (T4)**.

In [ ]:
!nvidia-smi

In [ ]:
# 1. Clone roboflow/sports and this repo
REPO_URL = "https://github.com/varunrout/broadcast-cv-pipeline.git"  # change if needed

%cd /content
!git clone -q https://github.com/roboflow/sports.git
!git clone -q {REPO_URL} pipeline

In [ ]:
# 2. Install
%cd /content/sports
!pip install -q -e .
%cd /content/sports/examples/soccer
!pip install -q -r requirements.txt
!pip install -q -r /content/pipeline/requirements.txt

In [ ]:
# 3. Download models and sample videos (~400 MB)
%cd /content/sports/examples/soccer
!bash setup.sh
!ls -lh data

In [ ]:
# 4. Trim to a 20-second clip
!ffmpeg -y -loglevel error -i /content/sports/examples/soccer/data/2e57b9_0.mp4 -t 20 -c copy /content/sports/examples/soccer/data/clip.mp4

In [ ]:
# 5. Run all modes and log runtimes
%cd /content/pipeline
!python src/run_modes.py --soccer-dir /content/sports/examples/soccer --clip data/clip.mp4 --out-dir outputs --device cuda

In [ ]:
# 6. fps and real-time ratio per mode
!python src/benchmark.py --clip /content/sports/examples/soccer/data/clip.mp4

In [ ]:
# 7. Re-encode outputs so they play inline
import glob, os
for src in sorted(glob.glob('/content/pipeline/outputs/out_*.mp4')):
    dst = src.replace('out_', 'view_')
    os.system(f'ffmpeg -y -loglevel error -i {src} -vcodec libx264 -preset veryfast -crf 28 {dst}')
!ls -lh /content/pipeline/outputs

In [ ]:
# 8. Watch one mode at a time
from IPython.display import Video
MODE_TO_VIEW = 'player_tracking'  # pitch_detection | player_tracking | team_classification | radar
Video(f'/content/pipeline/outputs/view_{MODE_TO_VIEW}.mp4', embed=True, width=800)

Fill in `results/failure_modes.md` while watching. Download `results/timings.csv` and `results/benchmark.csv` back into the repo.